# 01 — Catálogo y datos de evaluación

Preparamos el catálogo ESCI en español y separamos las consultas de desarrollo y evaluación. Conservamos todos los productos anotados de cada consulta; el agente recibe el catálogo sin etiquetas de relevancia.


In [5]:
%pip -q install -r requirements-lab4-modular.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements-lab4-modular.txt'
Note: you may need to restart the kernel to use updated packages.


In [6]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src" / "lab4_asistente").is_dir():
    ROOT = next((p for p in ROOT.parents if (p / "src" / "lab4_asistente").is_dir()), ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Raíz del proyecto:", ROOT)
import json
import pandas as pd
from lab4_asistente import config as C
from lab4_asistente.datos import preparar


Raíz del proyecto: /home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz


## Preparar el subconjunto de trabajo

Se usan 150 consultas `train` para desarrollo y 250 consultas `test` para la evaluación final. La selección se realiza por `query_id`; se conservan todos los pares anotados de las consultas elegidas y se agregan productos distractores hasta alcanzar el mínimo solicitado.

In [7]:
resumen_datos = preparar(
    C.RUTA_INPUT, C.RUTA_DATOS, C.RUTA_EVAL,
    n_eval=C.N_CONSULTAS_EVAL,
    n_dev=C.N_CONSULTAS_DEV,
    min_catalogo=C.MIN_CATALOGO,
    semilla=C.SEMILLA,
)
print(json.dumps(resumen_datos, indent=2, ensure_ascii=False))


{
  "productos": 10688,
  "distractores": 0,
  "consultas_eval": 250,
  "consultas_dev": 150,
  "pares_anotados": 10928,
  "etiquetas": {
    "dev": {
      "C": 305,
      "E": 1645,
      "I": 690,
      "S": 1399
    },
    "eval": {
      "C": 548,
      "E": 2814,
      "I": 1032,
      "S": 2495
    }
  },
  "faltantes_%": {
    "descripcion": 47.9,
    "caracteristicas": 12.0,
    "marca": 4.7,
    "color": 37.2
  }
}


## Comprobar separación y cobertura

In [8]:
catalogo = pd.read_parquet(C.RUTA_DATOS / "catalogo.parquet")
consultas = pd.read_parquet(C.RUTA_DATOS / "consultas.parquet")
qrels = pd.read_parquet(C.RUTA_EVAL / "qrels.parquet")

assert len(catalogo) >= C.MIN_CATALOGO
assert (consultas["conjunto"] == "eval").sum() >= 200
assert qrels["product_id"].isin(catalogo["product_id"]).all()
assert "esci_label" not in catalogo.columns

print("Catálogo:", len(catalogo))
print("Consultas dev/eval:", consultas["conjunto"].value_counts().to_dict())
print("Pares anotados:", len(qrels))
print("Etiquetas separadas del catálogo: OK")


Catálogo: 10688
Consultas dev/eval: {'eval': 250, 'dev': 150}
Pares anotados: 10928
Etiquetas separadas del catálogo: OK
